# Fans most favorites genre analysis

In [5]:
import duckdb  # <--- Importante: serve per eseguire la query
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Definisci i percorsi dei tuoi file
ratings_path = 'cleaned_data/ratings_cleaned.parquet'
profiles_path = 'cleaned_data/profiles_cleaned.parquet'

# 2. Definisci la Query (Nota: la chiamo 'query' per chiarezza)
query = f"""
    WITH user_scores AS (
        SELECT
            username,
            AVG(score) as mean_score
        FROM '{ratings_path}'
        GROUP BY username
    )
    SELECT
        p.username,
        p.completed,
        COALESCE(s.mean_score, 0) as mean_score
    FROM '{profiles_path}' p
    LEFT JOIN user_scores s ON p.username = s.username
    WHERE p.completed IS NOT NULL
"""

# 3. ESECUZIONE (Il passaggio che mancava)
# Eseguiamo la query con DuckDB e convertiamo il risultato in Pandas DataFrame
profiles = duckdb.sql(query).df()

# 4. Clustering Manuale con Pandas
# qcut divide i dati in 3 gruppi di uguale numerosità
profiles['activity_level'] = pd.qcut(
    profiles['completed'],
    q=3,
    labels=['Novizio', 'Appassionato', 'Veterano']
)

# 5. Vediamo le statistiche medie per ogni gruppo
# observed=True evita warning futuri se usi categorie
print(profiles.groupby('activity_level', observed=True)[['completed', 'mean_score']].mean())

IOException: IO Error: No files found that match the pattern "cleaned_data/ratings_cleaned.parquet"

### Supponiamo tu abbia un DataFrame 'user_genres' dove ogni riga è: Utente | Genere_Visto
 (Lo ottieni facendo il join tra Favs/Ratings e la tabella Anime che contiene i generi)


In [ ]:
# Esempio di logica con Pandas:
# 1. Conta quante volte un utente ha visto un certo genere
genre_counts = user_genres.groupby(['username', 'genre']).size().reset_index(name='count')

# 2. Trova l'indice del genere con il conteggio più alto per ogni utente
idx = genre_counts.groupby('username')['count'].idxmax()

# 3. Estrai la riga corrispondente
dominant_tastes = genre_counts.loc[idx]

# Ora hai una tabella che ti dice: UserA -> "Action", UserB -> "Romance"
dominant_tastes.rename(columns={'genre': 'main_genre'}, inplace=True)

# Uniscilo ai profili
profiles = profiles.merge(dominant_tastes[['username', 'main_genre']], on='username', how='left')

3. Visualizzazione dei Gruppi (Analisi Grafica)
Ora che hai assegnato le etichette manualmente, usa seaborn o plotly per vedere se questi gruppi hanno senso. Questo sostituisce visivamente il K-Means.

In [ ]:
# Grafico a dispersione: Quantità vs Qualità
plt.figure(figsize=(12, 8))

sns.scatterplot(
    data=profiles,
    x='completed',
    y='mean_score',
    hue='main_genre',  # Colora i punti in base al genere preferito
    style='activity_level', # Cambia la forma in base al livello di attività
    alpha=0.6
)

plt.xscale('log') # Utile perché alcuni utenti hanno 10 anime, altri 10.000
plt.title('Cluster Utenti: Attività vs Voto Medio (Colorati per Genere)')
plt.show()